# Lab 16 solution: the coding agent on a second project, with and without its AGENTS.md loaded

This notebook solves the three Challenge tasks of Lab 16. It repeats what it needs from the lab (the first cell, the
workspace copy, the path check, the four tools and their schemas, the loop and Part C's system message), so it runs
on its own. The project is `challenge_start/`: `totals.py`, its test and an `AGENTS.md` with two rules.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
import shutil
import subprocess
import sys
import textwrap
from openai import OpenAI, omit

client = OpenAI(timeout=60)
LAB = ROOT / "16_Coding_Agent"
WORK = LAB / "workspace"                  # the folder the agent works in; git ignores it
CHALLENGE = LAB / "challenge_start"       # the second project's start folder
MAX_OUTPUT = 800                          # tokens per reply

From the lab: `fresh_workspace` (Part A), the path check and the four tools (Parts C and D), each returning text:

In [3]:
def fresh_workspace(start):
    """Delete workspace/ and copy the start folder in its place."""
    shutil.rmtree(WORK, ignore_errors=True)
    shutil.copytree(start, WORK)


def check_path(name):
    """The file inside the workspace that name points to, or None when name leaves the workspace."""
    target = (WORK / name).resolve()
    if target.is_relative_to(WORK.resolve()):
        return target
    return None


def list_files():
    """The names of the files in the workspace, one per line."""
    return "\n".join(sorted(p.name for p in WORK.iterdir() if p.is_file()))


def read_file(path):
    """The text of one file in the workspace, or a refusal."""
    target = check_path(path)
    if target is None:
        return f"Refused: {path} is outside the workspace."
    if not target.is_file():
        return f"There is no file named {path} in the workspace."
    return target.read_text()


def write_file(path, content):
    """Replace the whole file with content, or refuse."""
    target = check_path(path)
    if target is None:
        return f"Refused: {path} is outside the workspace."
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}."


def run_tests():
    """Run the unit tests in the workspace and return the report as text."""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "unittest", "-v"],
            cwd=WORK, capture_output=True, text=True, timeout=30,
        )
    except subprocess.TimeoutExpired:
        return "The tests did not finish within 30 seconds and were stopped."
    return (result.stdout + result.stderr).replace(str(WORK.resolve()), "workspace")

The four schemas (Parts B, C and E), the map from tool names to functions, and Part C's system message:

In [4]:
def no_arguments(name, description):
    return {"type": "function", "function": {"name": name, "description": description, "strict": True,
            "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}}}


PATH = {"type": "string", "description": "A file name, for example 'todo.txt'"}
READ_SCHEMA = {"type": "function", "function": {
    "name": "read_file", "description": "Read one file in the folder.", "strict": True,
    "parameters": {"type": "object", "properties": {"path": PATH}, "required": ["path"],
                   "additionalProperties": False}}}
WRITE_SCHEMA = {"type": "function", "function": {
    "name": "write_file", "description": "Replace the whole content of one file in the folder (creates it if missing).",
    "strict": True,
    "parameters": {"type": "object", "properties": {"path": PATH, "content": {"type": "string"}},
                   "required": ["path", "content"], "additionalProperties": False}}}

TOOLS = [no_arguments("list_files", "List the files in the folder."), READ_SCHEMA, WRITE_SCHEMA,
         no_arguments("run_tests", "Run the unit tests in the folder and return their report.")]
FUNCTIONS = {"list_files": list_files, "read_file": read_file, "write_file": write_file, "run_tests": run_tests}
CODE_SYSTEM = ("You fix Python code in a small folder. Read the failing test and the code before you change "
               "anything, and run the tests after every change. When the tests pass, stop and say in one or two "
               "sentences what you changed.")
TASK = "The test in test_totals.py fails. Fix line_total in totals.py."

`no_arguments` writes the schema of a tool that takes no arguments, `list_files` and `run_tests`; the lab writes
both out in full. Then the loop from Part B, which prints one line per tool call and returns the message list:

In [5]:
def run_agent(system, task, tools, functions, max_turns=15):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            max_completion_tokens=MAX_OUTPUT, reasoning_effort=EFFORT or omit,
        )
        reply = response.choices[0].message
        messages.append(reply)
        if response.choices[0].finish_reason != "tool_calls":
            print(f"turn {turn}: answer: {reply.content}")
            return messages
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments)
            result = functions[call.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
            print(f"turn {turn}: {call.function.name}({args.get('path', '')})")
            if call.function.name == "write_file":
                print(textwrap.indent(args["content"].rstrip(), "    "))
            if call.function.name == "run_tests":
                report = [line for line in result.splitlines() if line.startswith(("AssertionError", "OK", "FAILED"))]
                print(textwrap.indent("\n".join(report), "    "))
    print("turn cap reached: no answer")
    return messages

---

## Task 1: Run the agent on the second project, with its AGENTS.md loaded

A fresh copy of `challenge_start/`, its files and its tests before any agent runs:

In [6]:
fresh_workspace(CHALLENGE)
print(list_files())
print("-----")
print(run_tests())

AGENTS.md
test_totals.py
totals.py
-----
test_one_track (test_totals.LineTotalTest.test_one_track) ... FAIL
test_two_tracks (test_totals.LineTotalTest.test_two_tracks) ... FAIL

FAIL: test_one_track (test_totals.LineTotalTest.test_one_track)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "workspace/test_totals.py", line 8, in test_one_track
    self.assertEqual(line_total(1.99, 1), 1.99)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 2.99 != 1.99

FAIL: test_two_tracks (test_totals.LineTotalTest.test_two_tracks)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "workspace/test_totals.py", line 11, in test_two_tracks
    self.assertEqual(line_total(0.99, 2), 1.98)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 2.99 != 1.98

----------------------------------------------------------------------
Ran 2 tests in 0.000s

FAILED (fa

Three files, one of them `AGENTS.md` with the two rules. Both tests fail: `line_total` adds, so 1.99 + 1 and
0.99 + 2 both give 2.99 instead of 1.99 and 1.98. No test looks at `CHANGES.md`.

The system message is built as in Part E: Part C's instructions, then the project's `AGENTS.md` word for word. One
run, from a fresh copy:

In [7]:
fresh_workspace(CHALLENGE)
rules = (WORK / "AGENTS.md").read_text()
SYSTEM_WITH_RULES = CODE_SYSTEM + "\n\nThe workspace's AGENTS.md says:\n" + rules
messages = run_agent(SYSTEM_WITH_RULES, TASK, TOOLS, FUNCTIONS)

turn 1: list_files()
turn 1: read_file(test_totals.py)
turn 1: read_file(totals.py)
turn 1: read_file(AGENTS.md)
turn 1: read_file(CHANGES.md)


turn 2: write_file(totals.py)
    def line_total(unit_price, quantity):
        """Total of one invoice line: a track's price times how many were bought."""
        return unit_price * quantity


turn 3: run_tests()
    OK
turn 3: write_file(CHANGES.md)
    - totals.py, line_total: multiply unit price by quantity to calculate the line total.


turn 4: answer: Updated `line_total` to multiply the unit price by the quantity, and recorded the fix in `CHANGES.md`. All tests pass.


In its first turn the agent asked for five things at once: the file list, the test, the code, `AGENTS.md`
(although its rules were already in the system message) and `CHANGES.md`, which does not exist yet, so our
`read_file` answered that one with a sentence. It wrote `unit_price * quantity`, then ran the tests (`OK`) and wrote
one line in `CHANGES.md` in the same reply, and answered in its fourth turn. No rounding was needed this time:
`0.99 * 2` is `1.98` in Python.

The disk, checked as in Part F: the patch, our own test run, the test file against the start folder's copy, and
`CHANGES.md`.

In [8]:
import difflib

before = (CHALLENGE / "totals.py").read_text().splitlines(keepends=True)
after = (WORK / "totals.py").read_text().splitlines(keepends=True)
print("".join(difflib.unified_diff(before, after, "challenge_start/totals.py", "workspace/totals.py")))
print(run_tests())
print("test_totals.py unchanged:", (WORK / "test_totals.py").read_text() == (CHALLENGE / "test_totals.py").read_text())
changes = WORK / "CHANGES.md"
print("CHANGES.md:", changes.read_text().strip() if changes.exists() else None)

--- challenge_start/totals.py
+++ workspace/totals.py
@@ -1,3 +1,3 @@
 def line_total(unit_price, quantity):
     """Total of one invoice line: a track's price times how many were bought."""
-    return unit_price + quantity
+    return unit_price * quantity

test_one_track (test_totals.LineTotalTest.test_one_track) ... ok
test_two_tracks (test_totals.LineTotalTest.test_two_tracks) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK

test_totals.py unchanged: True
CHANGES.md: - totals.py, line_total: multiply unit price by quantity to calculate the line total.


One line changed, `unit_price + quantity` to `unit_price * quantity`; our own run ends in `OK`;
`test_totals.py unchanged: True`; and `CHANGES.md` holds the line the rule asks for: the file, the function and
what changed.

---

## Task 2: Leave AGENTS.md in the folder, and do not load it

Now the system message is `CODE_SYSTEM` alone. `AGENTS.md` is still in the folder, so the model will see its name
if it calls `list_files`, and nothing more.

To tell whether the agent opened the file, we look through the messages `run_agent` returned. The dictionaries are
our own messages (the system message, the task and the tool results), so we skip them; every other item is a reply
from the model, and its `tool_calls` (or `None`) say what it asked for. `opened` returns `True` when one of those
calls was `read_file` on a path ending in the file name, so `./AGENTS.md` counts too.

In [9]:
def opened(messages, name):
    """True when one of the model's replies asked read_file for this file."""
    for message in messages:
        if isinstance(message, dict):
            continue
        for call in message.tool_calls or []:
            if call.function.name == "read_file" and json.loads(call.function.arguments)["path"].endswith(name):
                return True
    return False


print("Task 1's run opened AGENTS.md:", opened(messages, "AGENTS.md"))

Task 1's run opened AGENTS.md: True


`True`: in Task 1 the agent opened `AGENTS.md` itself, in its first turn, although the rules were already in
its system message.

Three runs, each from a fresh copy, with the rules left in the folder:

In [10]:
for number in range(1, 4):
    print(f"===== run {number}")
    fresh_workspace(CHALLENGE)
    messages = run_agent(CODE_SYSTEM, TASK, TOOLS, FUNCTIONS)
    same_test = (WORK / "test_totals.py").read_text() == (CHALLENGE / "test_totals.py").read_text()
    changes = WORK / "CHANGES.md"
    print("our test run:", run_tests().rstrip().splitlines()[-1], "| test unchanged:", same_test,
          "| opened AGENTS.md:", opened(messages, "AGENTS.md"))
    print("CHANGES.md:", changes.read_text().strip() if changes.exists() else None)

===== run 1


turn 1: list_files()


turn 2: read_file(test_totals.py)
turn 2: read_file(totals.py)


turn 3: run_tests()
    AssertionError: 2.99 != 1.99
    AssertionError: 2.99 != 1.98
    FAILED (failures=2)


turn 4: write_file(totals.py)
    def line_total(unit_price, quantity):
        """Total of one invoice line: a track's price times how many were bought."""
        return unit_price * quantity


turn 5: run_tests()
    OK


... (8 outputs left out)

turn 2: read_file(test_totals.py)
turn 2: read_file(totals.py)
turn 2: read_file(AGENTS.md)


turn 3: write_file(totals.py)
    def line_total(unit_price, quantity):
        """Total of one invoice line: a track's price times how many were bought."""
        return unit_price * quantity


turn 4: run_tests()
    OK


turn 5: answer: Fixed `line_total` to multiply the unit price by the quantity. Both tests pass.
our test run: OK | test unchanged: True | opened AGENTS.md: True
CHANGES.md: None


All three runs fixed `line_total`, and our own test run said `OK` with the test unchanged. **None wrote
`CHANGES.md`** (0 of 3), and only one opened `AGENTS.md` (1 of 3). Runs 1 and 2 listed the files, read the test and
the code, and went straight to work. Run 3 opened `AGENTS.md` in the same turn as the code and still skipped the
rule. Every answer says the tests pass, which is true, and says nothing about the rule. How often an agent opens a
rules file it was not given, and then follows it, varies from run to run: three runs of yours can show anything
from none to all.

---

## Task 3: What loading AGENTS.md changed

- **Who makes sure the rules are in front of the model.** When the rules are loaded, our code puts them into the
  instructions before the first turn, in every run. When they are only left in the folder, the model decides, run
  by run, whether to open the file, and then whether to follow it. That is why coding agents such as Codex read
  `AGENTS.md` with their own code before the model is called.
- **Why a rule that no test checks needs the loaded version.** The tests catch a wrong `line_total`, so the loop
  corrects it whether or not the rules were read. Nothing catches a missing `CHANGES.md` line: the run still ends
  with passing tests and a reply that says the work is done. A rule with no check behind it holds only when it is
  in the instructions, and even then only the disk tells you it was followed.
- **Loading the rules does not make the model follow them.** It can still skip a line in its instructions. In
  Task 1's run it did not, but only the check of `CHANGES.md` on disk tells you that, so keep it.